## Community experiment: distributed migration go/no-go

This notebook runs a **distributed** AutoGen Core setup: a **gRPC host** plus one or more **GrpcWorkerAgentRuntime** workers, with agents registered as **`RoutedAgent`** subclasses that delegate to **`AssistantAgent`**.

### What it does

- **Scenario:** Decide whether to proceed with a planned database-related change (e.g. major Postgres upgrade or search reindex).
- **MigrationAdvocate:** Argues for going ahead on schedule; may call a **`FunctionTool`** that returns static **service impact** facts.
- **RiskOfficer:** Argues for caution; may call a **`FunctionTool`** that returns an internal **migration rubric** excerpt.
- **EngineeringLead:** Fans out to both specialists using **`asyncio.gather`** (parallel messaging), then produces a structured **`MigrationDecision`** (GO / NO-GO / DEFER, rationale, confidence, follow-ups) via **Pydantic** `response_format`.
- **MigrationBrief:** Scenarios are described with a small dataclass and **named presets** you can switch with `ACTIVE_SCENARIO`.
- **Deployment shape:** Toggle **`ALL_IN_ONE_WORKER`** to run every agent in one worker process or split advocate / risk / lead across workers.

**Requirements:** `OPENAI_API_KEY` in `.env`. Default **`GRPC_HOST`** is **`localhost:50053`**; change it if that port is already in use on your machine.

In [1]:
import asyncio
import json
import time
from dataclasses import dataclass
from typing import Literal

from pydantic import BaseModel, Field, ValidationError

from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_core.tools import FunctionTool
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from IPython.display import Markdown, display
from dotenv import load_dotenv

load_dotenv(override=True)

ALL_IN_ONE_WORKER = False
GRPC_HOST = "localhost:50053"

### Message type

In [2]:
@dataclass
class Message:
    content: str

### gRPC host

In [3]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost

host = GrpcWorkerAgentRuntimeHost(address=GRPC_HOST)
host.start()

### Structured migration brief + scenario presets

In [4]:
@dataclass(frozen=True)
class MigrationBrief:
    """Structured change record — easier to swap scenarios than one long string."""

    title: str
    service_key: str
    technical_summary: str
    maintenance_window_utc: str
    blast_radius: str
    rollback_ready: bool

    def as_prompt_block(self) -> str:
        return (
            f"### {self.title}\n"
            f"- **Blast radius:** {self.blast_radius}\n"
            f"- **Window (UTC):** {self.maintenance_window_utc}\n"
            f"- **Rollback tested on staging clone:** {self.rollback_ready}\n"
            f"- **Technical summary:** {self.technical_summary}\n"
        )


SCENARIOS: dict[str, MigrationBrief] = {
    "billing_pg_upgrade": MigrationBrief(
        title="PostgreSQL major upgrade (billing shard)",
        service_key="billing_pg",
        technical_summary=(
            "13 → 16 on primary billing shard; ~45 min read-only; blue/green app cutover after vacuum/analyze."
        ),
        maintenance_window_utc="Saturday 02:00–06:00",
        blast_radius="Billing API + invoice pipeline; no analytics warehouse",
        rollback_ready=True,
    ),
    "search_reindex": MigrationBrief(
        title="Elasticsearch full cluster reindex",
        service_key="search_es",
        technical_summary=(
            "Reindex 12B docs to new mapping; dual-write already on; cut traffic via feature flag."
        ),
        maintenance_window_utc="Sunday 01:00–05:00",
        blast_radius="Product search + autocomplete; checkout unaffected",
        rollback_ready=False,
    ),
}

# Flip this key to try the second scenario without editing prose.
ACTIVE_SCENARIO = "billing_pg_upgrade"
brief = SCENARIOS[ACTIVE_SCENARIO]


def advocate_prompt(b: MigrationBrief) -> str:
    return (
        f"{b.as_prompt_block()}\n\n"
        "Argue clearly FOR executing this change on schedule. "
        "You may call `get_service_impact_facts` once with the service_key from the brief to cite static impact facts. "
        "Then finish with your argument. Keep under ~250 words; bullets OK. "
        "Do not end with the word TERMINATE."
    )


def risk_prompt(b: MigrationBrief) -> str:
    return (
        f"{b.as_prompt_block()}\n\n"
        "Argue for caution: failure modes, rollback limits, verification gaps, team readiness. "
        "You may call `get_org_migration_rubric` once to ground your thinking in the internal checklist. "
        "Then finish. Keep under ~250 words; bullets OK. "
        "Do not end with the word TERMINATE."
    )

### Local `FunctionTool`s (deterministic — unlike Serper in the lab)

In [5]:
def get_service_impact_facts(service_key: str) -> str:
    """Static org facts for planning (no network)."""
    facts = {
        "billing_pg": (
            "Tier-1 revenue path; ~2.1M invoices/day peak; downstream: invoices API, tax, dunning. "
            "Error budget: 0.05% monthly bad minutes. Finance freeze week after quarter close — not this window."
        ),
        "search_es": (
            "Tier-2 discovery; session revenue lift ~3% from search; no direct payment path. "
            "Degraded mode: hide suggestions, core search stays up on old index."
        ),
    }
    return facts.get(service_key, f"Unknown service_key={service_key!r}; use billing_pg or search_es.")


def get_org_migration_rubric() -> str:
    """Fake but stable compliance snippet the risk agent can cite."""
    return (
        "Org DB migration rubric (excerpt):\n"
        "- [ ] Rollback script replayed on staging clone\n"
        "- [ ] On-call: primary + shadow from data platform\n"
        "- [ ] Feature flags for read-only mode tested\n"
        "- [ ] Customer comms queued ≥72h before Tier-1\n"
        "- [ ] Post-cutover: 2h extended monitoring + synthetic checks\n"
    )


impact_facts_tool = FunctionTool(
    get_service_impact_facts,
    description="Return static service impact / dependency facts for a known service_key.",
    strict=True,
)

rubric_tool = FunctionTool(
    get_org_migration_rubric,
    description="Return the internal checklist excerpt for production database changes.",
    strict=True,
)


class MigrationDecision(BaseModel):
    decision: Literal["GO", "NO-GO", "DEFER"] = Field(description="Single outcome label")
    rationale: str = Field(description="2–5 sentences, only using specialist input below")
    confidence: int = Field(ge=0, le=100, description="Confidence 0–100")
    follow_up_actions: list[str] = Field(
        default_factory=list,
        max_length=5,
        description="Up to 5 concrete next steps (e.g. load test, extend window)",
    )


LEAD_SYSTEM = (
    "You are the engineering lead. You receive two specialist write-ups (for / cautious). "
    "Respond ONLY with JSON that matches the provided schema. No markdown, no TERMINATE. "
    "Do not invent external facts beyond what the specialists wrote (the tools they used are already reflected in their text)."
)


def format_decision_markdown(combined_human: str, raw_json: str, validated: MigrationDecision | None) -> str:
    parts = [combined_human, "## Structured decision (Pydantic)\n"]
    if validated is not None:
        parts.append(f"- **Outcome:** `{validated.decision}`  \n")
        parts.append(f"- **Confidence:** {validated.confidence}/100  \n")
        parts.append(f"- **Rationale:** {validated.rationale}  \n")
        if validated.follow_up_actions:
            parts.append("- **Follow-ups:**\n")
            for a in validated.follow_up_actions:
                parts.append(f"  - {a}\n")
    else:
        parts.append(f"_Could not parse structured output; raw:\n```json\n{raw_json}\n```\n")
    return "".join(parts)

### Routed agents

In [6]:
class MigrationAdvocate(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(
            name,
            model_client=model_client,
            tools=[impact_facts_tool],
            reflect_on_tool_use=True,
        )

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)


class RiskOfficer(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(
            name,
            model_client=model_client,
            tools=[rubric_tool],
            reflect_on_tool_use=True,
        )

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)


class EngineeringLead(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(
            model="gpt-4o-mini",
            temperature=0.2,
            response_format=MigrationDecision,
        )
        self._delegate = AssistantAgent(name, model_client=model_client, system_message=LEAD_SYSTEM)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        inner_advocate = AgentId("advocate", "default")
        inner_risk = AgentId("risk", "default")
        m_adv = Message(content=advocate_prompt(brief))
        m_risk = Message(content=risk_prompt(brief))

        t0 = time.perf_counter()
        response_adv, response_risk = await asyncio.gather(
            self.send_message(m_adv, inner_advocate),
            self.send_message(m_risk, inner_risk),
        )
        elapsed = time.perf_counter() - t0
        print(f"Specialist round (parallel gather): {elapsed:.2f}s")

        combined = (
            f"## Case for migrating\n{response_adv.content}\n\n"
            f"## Risks and caution\n{response_risk.content}\n\n"
        )
        user_prompt = (
            f"{combined}\n"
            "Produce the MigrationDecision JSON based solely on the two sections above."
        )
        text_message = TextMessage(content=user_prompt, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        raw = response.chat_message.content
        validated: MigrationDecision | None = None
        try:
            payload = json.loads(raw) if isinstance(raw, str) else raw
            validated = MigrationDecision.model_validate(payload)
        except (json.JSONDecodeError, ValidationError, TypeError):
            pass
        out = format_decision_markdown(combined, raw if isinstance(raw, str) else json.dumps(raw), validated)
        return Message(content=out)

### Register workers

In [7]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:
    worker = GrpcWorkerAgentRuntime(host_address=GRPC_HOST)
    await worker.start()
    await MigrationAdvocate.register(worker, "advocate", lambda: MigrationAdvocate("advocate"))
    await RiskOfficer.register(worker, "risk", lambda: RiskOfficer("risk"))
    await EngineeringLead.register(worker, "lead", lambda: EngineeringLead("lead"))
    agent_id = AgentId("lead", "default")
else:
    worker_adv = GrpcWorkerAgentRuntime(host_address=GRPC_HOST)
    await worker_adv.start()
    await MigrationAdvocate.register(worker_adv, "advocate", lambda: MigrationAdvocate("advocate"))

    worker_risk = GrpcWorkerAgentRuntime(host_address=GRPC_HOST)
    await worker_risk.start()
    await RiskOfficer.register(worker_risk, "risk", lambda: RiskOfficer("risk"))

    worker = GrpcWorkerAgentRuntime(host_address=GRPC_HOST)
    await worker.start()
    await EngineeringLead.register(worker, "lead", lambda: EngineeringLead("lead"))
    agent_id = AgentId("lead", "default")

Specialist round (parallel gather): 6.87s


### Run the lead agent (parallel specialist fan-out inside `EngineeringLead`)

In [8]:
response = await worker.send_message(Message(content="Decide on the migration."), agent_id)
display(Markdown(response.content))

## Case for migrating
- **Scheduled Maintenance Window**: The upgrade is aligned with non-peak hours (Saturday, 02:00–06:00 UTC), minimizing potential user disruption.
  
- **Rollback Preparedness**: The rollback has been tested successfully on a staging clone, ensuring that we can revert changes quickly if unforeseen issues arise.

- **Enhanced Performance**: Upgrading PostgreSQL from version 13 to 16 will likely improve performance, scalability, and security, resulting in better operational efficiency for the billing API and invoice pipeline.

- **Read-Only Period**: The planned 45 minutes in a read-only state is manageable and clearly communicated; therefore, users will have advance notice to handle any dependencies on the billing system.

- **Low Impact on Other Services**: The operation specifically mentions that the analytics warehouse will not be impacted, thus limiting the effects to the billing API and reducing the overall system risk.

- **Post-Upgrade Maintenance**: The follow-up vacuum and analyze processes will ensure the database is optimized immediately after the upgrade, safeguarding against performance regressions.

In conclusion, executing this PostgreSQL upgrade on schedule presents a low-risk, high-reward opportunity for enhancing our billing infrastructure while ensuring adequate preparations are in place to manage any contingencies effectively.

## Risks and caution
When considering the PostgreSQL upgrade from version 13 to 16 for the billing shard, several cautionary points warrant discussion:

- **Failure Modes:** The potential for database corruption or migration errors during the upgrade could result in significant data loss or service disruption, especially affecting billing operations.

- **Rollback Limits:** While rollback has been tested on a staging clone, the live environment may introduce variables that could complicate recovery. The rollback scripts need to be flawless and fail-proof, which still poses risks if unforeseen issues arise.

- **Verification Gaps:** Post-migration, relying solely on manual or limited checks could leave blind spots in verifying data integrity. Ensuring comprehensive synthetic checks and monitoring is critical to affirming successful upgrades.

- **Team Readiness:** There needs to be a clear understanding and readiness among the team regarding the upgrade plan, including each member's role. Lack of preparedness or contingencies could exacerbate issues or prolong downtime.

Given the potential impact on the billing API and invoice pipeline, thorough preparations must be ensured, including appropriate customer communications and extended post-cutover monitoring as per migration standards. 

This upgrade needs careful orchestration to safeguard against any repercussions while diligently ensuring system integrity.

## Structured decision (Pydantic)
- **Outcome:** `DEFER`  
- **Confidence:** 70/100  
- **Rationale:** While the upgrade presents a low-risk, high-reward opportunity, there are significant concerns regarding potential database corruption and migration errors that could lead to data loss. The rollback process, although tested, may face complications in the live environment, and there are gaps in verification that could leave data integrity unconfirmed. Additionally, team readiness is crucial, and any lack of preparedness could exacerbate issues during the migration.  
- **Follow-ups:**
  - Conduct a detailed risk assessment
  - Enhance rollback scripts for live environment
  - Implement comprehensive synthetic checks
  - Ensure team training and readiness
  - Communicate with customers about potential impacts


### Shutdown

In [9]:
await worker.stop()
if not ALL_IN_ONE_WORKER:
    await worker_adv.stop()
    await worker_risk.stop()

In [ ]:
await host.stop()